In [ ]:
# ============================================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# ============================================================
import os
import sys
import json
import time
import shutil
import random
import logging
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    ssd300_vgg16,
    retinanet_resnet50_fpn_v2
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.ssd import SSDClassificationHead

from sklearn.metrics import confusion_matrix
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# ──────────── GPU Setup ────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    logger.info(f"✅ GPU: {gpu_name} | VRAM: {gpu_mem:.1f} GB")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    logger.warning("⚠️ GPU not found — running on CPU")

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__} | Torchvision: {torchvision.__version__}")


In [ ]:
# ============================================================
# CELL 2: PATHS & HYPERPARAMETERS
# ============================================================

# ──────────── Root paths ────────────
BDD_ROOT = Path(r"D:\Data\rdd2022")
TRAIN_IMG_DIR   = BDD_ROOT / "train" / "images"
TRAIN_LBL_DIR   = BDD_ROOT / "train" / "labels"
VALID_IMG_DIR   = BDD_ROOT / "valid" / "images"
VALID_LBL_DIR   = BDD_ROOT / "valid" / "labels"
TEST_IMG_DIR    = BDD_ROOT / "test"  / "images"
TEST_LBL_DIR    = BDD_ROOT / "test"  / "labels"

# Output dirs
OUTPUT_DIR      = BDD_ROOT / "outputs"
CKPT_DIR        = OUTPUT_DIR / "checkpoints"
LOG_DIR         = OUTPUT_DIR / "logs"
RESULT_DIR      = OUTPUT_DIR / "results"
for d in [OUTPUT_DIR, CKPT_DIR, LOG_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ──────────── Class mapping ────────────
# RDD2022: 4 damage types (PASCAL VOC label names)
CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
CLASS_MAP   = {"D00": 0, "D10": 1, "D20": 2, "D40": 3}
NUM_CLASSES = len(CLASS_NAMES) + 1  # +1 for background (torchvision convention)

# ──────────── Data split ratios ────────────
TRAIN_RATIO = 0.70
VALID_RATIO = 0.20
TEST_RATIO  = 0.10

# ──────────── Hyperparameters ────────────
# Optimised for RTX 5060 8GB + RDD2022 scale
HPARAMS = {
    # Training
    "num_epochs":      10,          # Exploratory 10-epoch run before model selection
    "batch_size":      8,           # Safe for 8 GB VRAM with 800×800 images
    "num_workers":     4,
    "pin_memory":      True,
    # Optimiser
    "optimizer":       "AdamW",
    "learning_rate":   1e-4,        # Good starting LR for fine-tuning pretrained detectors
    "weight_decay":    1e-4,
    "momentum":        0.9,         # Used by SGD fallback
    # Scheduler
    "scheduler":       "CosineAnnealingLR",
    "lr_min":          1e-6,
    # Input
    "img_size":        800,         # Standard for FPN-based detectors
    # Augmentation
    "aug_prob":        0.5,
    # Misc
    "seed":            42,
    "amp":             True,        # Mixed precision (FP16) → faster on RTX 50xx
    "grad_clip":       5.0,
}

# ──────────── Reproducibility ────────────
random.seed(HPARAMS["seed"])
np.random.seed(HPARAMS["seed"])
torch.manual_seed(HPARAMS["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(HPARAMS["seed"])

# ──────────── Log hyperparameters ────────────
hparam_path = LOG_DIR / "hyperparameters.json"
with open(hparam_path, "w") as f:
    json.dump({**HPARAMS, "num_classes": NUM_CLASSES, "class_names": CLASS_NAMES}, f, indent=4)
logger.info(f"📝 Hyperparameters saved → {hparam_path}")
print("\n=== HYPERPARAMETERS ===")
for k, v in HPARAMS.items():
    print(f"  {k:25s}: {v}")


In [ ]:
# CELL 4: KIỂM TRA PHÂN PHỐI CLASS SAU KHI CHIA
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from pathlib import Path

CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}

def count_classes_in_split(img_dir: Path, lbl_dir: Path) -> Counter:
    """Đếm số instance từng class trong 1 split."""
    counter = Counter()
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    for img_p in img_dir.iterdir():
        if img_p.suffix.lower() not in exts:
            continue
        txt_p = lbl_dir / f"{img_p.stem}.txt"
        xml_p = lbl_dir / f"{img_p.stem}.xml"
        if txt_p.exists():
            with open(txt_p) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        counter[int(parts[0])] += 1
        elif xml_p.exists():
            ann = parse_xml_annotation(xml_p)
            for lbl in ann["labels"]:
                counter[lbl] += 1
    return counter

# ── Đếm từng split ──
splits = {
    "Train (70%)": (TRAIN_IMG_DIR, TRAIN_LBL_DIR),
    "Valid (20%)": (VALID_IMG_DIR, VALID_LBL_DIR),
    "Test  (10%)": (TEST_IMG_DIR,  TEST_LBL_DIR),
}

counts = {}
for split_name, (img_d, lbl_d) in splits.items():
    counts[split_name] = count_classes_in_split(img_d, lbl_d)

# ── In bảng số liệu ──
print(f"\n{'='*65}")
print(f"{'Class':<8} {'Train':>10} {'Train%':>8} {'Valid':>10} {'Valid%':>8} {'Test':>10} {'Test%':>8}")
print(f"{'-'*65}")

split_keys = list(counts.keys())
for cls_id, cls_name in CLASS_NAMES.items():
    row = f"{cls_name:<8}"
    for sk in split_keys:
        total_in_split = sum(counts[sk].values())
        cnt = counts[sk].get(cls_id, 0)
        pct = cnt / total_in_split * 100 if total_in_split > 0 else 0
        row += f" {cnt:>10} {pct:>7.1f}%"
    print(row)

print(f"{'-'*65}")
for sk in split_keys:
    total = sum(counts[sk].values())
    print(f"{'TOTAL':<8} {total:>10}  (split: {sk})")
print(f"{'='*65}")

# ── Biểu đồ 1: Số lượng instance theo class ── 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Phân phối Class theo từng Split", fontsize=15, fontweight="bold")

colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]

for ax, (split_name, cnt) in zip(axes, counts.items()):
    labels = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES))]
    values = [cnt.get(i, 0) for i in range(len(CLASS_NAMES))]
    bars = ax.bar(labels, values, color=colors, edgecolor="white", linewidth=1.2)
    ax.bar_label(bars, fmt="%d", fontsize=10, padding=3)
    ax.set_title(split_name, fontsize=13, fontweight="bold")
    ax.set_ylabel("Số instance")
    ax.set_ylim(0, max(values) * 1.2)
    ax.grid(axis="y", alpha=0.3)
    # Thêm % lên mỗi cột
    total = sum(values)
    for bar, val in zip(bars, values):
        pct = val / total * 100 if total > 0 else 0
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.03,
                f"{pct:.1f}%", ha="center", va="bottom", fontsize=9, color="gray")

plt.tight_layout()
plt.savefig(RESULT_DIR / "class_dist_per_split.png", dpi=150)
plt.show()

# ── Biểu đồ 2: Tỉ lệ % class (so sánh 3 split cạnh nhau) ──
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(CLASS_NAMES))
width = 0.25
split_colors = ["#2980b9", "#27ae60", "#e67e22"]

for i, (split_name, cnt) in enumerate(counts.items()):
    total = sum(cnt.values())
    pcts  = [cnt.get(j, 0) / total * 100 if total > 0 else 0
             for j in range(len(CLASS_NAMES))]
    bars = ax.bar(x + i * width, pcts, width,
                  label=split_name, color=split_colors[i], alpha=0.85)
    ax.bar_label(bars, fmt="%.1f%%", fontsize=8, padding=2)

ax.set_xticks(x + width)
ax.set_xticklabels([CLASS_NAMES[j] for j in range(len(CLASS_NAMES))], fontsize=12)
ax.set_ylabel("Tỉ lệ (%)")
ax.set_ylim(0, 50)
ax.set_title("So sánh tỉ lệ % class giữa Train / Valid / Test", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.axhline(25, color="red", linestyle="--", alpha=0.4, label="Balanced (25%)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / "class_ratio_comparison.png", dpi=150)
plt.show()

# ── Đánh giá tự động ──
print("\n📊 ĐÁNH GIÁ PHÂN PHỐI:")
for cls_id, cls_name in CLASS_NAMES.items():
    pcts = []
    for sk in split_keys:
        total = sum(counts[sk].values())
        pcts.append(counts[sk].get(cls_id, 0) / total * 100 if total > 0 else 0)
    deviation = max(pcts) - min(pcts)
    status = "✅ Đều" if deviation < 5 else "⚠️  Lệch"
    print(f"  {cls_name}: Train={pcts[0]:.1f}% | Valid={pcts[1]:.1f}% | Test={pcts[2]:.1f}% | Lệch={deviation:.1f}%  {status}")


In [ ]:
# ============================================================
# CELL: KIỂM TRA % DỮ LIỆU TRÊN 3 TẬP (ảnh + nhãn)
# ============================================================

from pathlib import Path
from collections import Counter

EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def count_split(img_dir: Path, lbl_dir: Path):
    imgs  = [p for p in img_dir.iterdir() if p.suffix.lower() in EXTS]
    total = len(imgs)
    has_label = sum(
        1 for p in imgs
        if (lbl_dir / f"{p.stem}.txt").exists()
        or (lbl_dir / f"{p.stem}.xml").exists()
    )
    no_label = total - has_label
    return total, has_label, no_label

# ── Đếm ──
t_img, t_lbl, t_no = count_split(TRAIN_IMG_DIR, TRAIN_LBL_DIR)
v_img, v_lbl, v_no = count_split(VALID_IMG_DIR,  VALID_LBL_DIR)
e_img, e_lbl, e_no = count_split(TEST_IMG_DIR,   TEST_LBL_DIR)
grand_total = t_img + v_img + e_img

# ── Bảng số liệu ──
print(f"\n{'='*62}")
print(f"{'Split':<10} {'Ảnh':>8} {'%Tổng':>8} {'Có nhãn':>10} {'Thiếu nhãn':>12}")
print(f"{'-'*62}")
for name, n_img, n_lbl, n_no in [
    ("Train",  t_img, t_lbl, t_no),
    ("Valid",  v_img, v_lbl, v_no),
    ("Test",   e_img, e_lbl, e_no),
]:
    pct = n_img / grand_total * 100 if grand_total > 0 else 0
    print(f"{name:<10} {n_img:>8} {pct:>7.1f}%  {n_lbl:>10} {n_no:>12}")
print(f"{'-'*62}")
print(f"{'TỔNG':<10} {grand_total:>8} {'100.0%':>8}  {t_lbl+v_lbl+e_lbl:>10} {t_no+v_no+e_no:>12}")
print(f"{'='*62}")

# ── Đánh giá tự động ──
print("\n📊 ĐÁNH GIÁ:")
target = {"Train": 0.70, "Valid": 0.20, "Test": 0.10}
actual = {"Train": t_img, "Valid": v_img, "Test": e_img}
all_ok = True
for name, tgt in target.items():
    pct = actual[name] / grand_total if grand_total > 0 else 0
    diff = abs(pct - tgt) * 100
    status = "✅" if diff < 3 else "⚠️  LỆCH"
    print(f"  {name:<6}: {pct*100:.1f}%  (target {tgt*100:.0f}%)  |  lệch {diff:.1f}%  {status}")
    if diff >= 3:
        all_ok = False

if all_ok:
    print("\n✅ Tỉ lệ đạt chuẩn 70/20/10!")
else:
    print("\n⚠️  Tỉ lệ lệch — cần chạy lại Cell 4 để chia lại!")

# ── Biểu đồ Pie ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Phân bổ dữ liệu trên 3 tập", fontsize=14, fontweight="bold")

labels = ["Train", "Valid", "Test"]
sizes  = [t_img, v_img, e_img]
colors = ["#2980b9", "#27ae60", "#e67e22"]
explode = (0.03, 0.03, 0.03)

# Pie chart
axes[0].pie(
    sizes, labels=labels, colors=colors, explode=explode,
    autopct=lambda p: f"{p:.1f}%\n({int(p*grand_total/100):,})",
    startangle=140, textprops={"fontsize": 11}
)
axes[0].set_title("Tỉ lệ số ảnh", fontsize=12)

# Bar chart so sánh target vs actual
x = range(3)
target_vals = [0.70, 0.20, 0.10]
actual_vals = [s/grand_total for s in sizes]
w = 0.35
b1 = axes[1].bar([i - w/2 for i in x], [v*100 for v in target_vals],
                  w, label="Target", color="#bdc3c7", edgecolor="gray")
b2 = axes[1].bar([i + w/2 for i in x], [v*100 for v in actual_vals],
                  w, label="Actual", color=colors, edgecolor="white")
axes[1].bar_label(b1, fmt="%.0f%%", fontsize=9, padding=2)
axes[1].bar_label(b2, fmt="%.1f%%", fontsize=9, padding=2)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels, fontsize=12)
axes[1].set_ylabel("Tỉ lệ (%)")
axes[1].set_ylim(0, 90)
axes[1].set_title("Target vs Actual", fontsize=12)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / "split_ratio_check.png", dpi=150)
plt.show()


In [ ]:
# ============================================================
# CELL: CHIA LẠI DỮ LIỆU 70/20/10 → LƯU THƯ MỤC MỚI
# ============================================================

import shutil, random
from pathlib import Path
from sklearn.model_selection import train_test_split

EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

# ── 1. ĐƯỜNG DẪN GỐC (giữ nguyên) ──────────────────────────
SRC_SPLITS = [
    (TRAIN_IMG_DIR, TRAIN_LBL_DIR),
    (VALID_IMG_DIR, VALID_LBL_DIR),
    (TEST_IMG_DIR,  TEST_LBL_DIR),
]

# ── 2. THƯ MỤC ĐẦU RA MỚI ───────────────────────────────────
NEW_ROOT = BDD_ROOT.parent / "dataset_split_7020_10"   # ← thư mục MỚI
NEW_TRAIN_IMG = NEW_ROOT / "train" / "images"
NEW_TRAIN_LBL = NEW_ROOT / "train" / "labels"
NEW_VALID_IMG = NEW_ROOT / "valid" / "images"
NEW_VALID_LBL = NEW_ROOT / "valid" / "labels"
NEW_TEST_IMG  = NEW_ROOT / "test"  / "images"
NEW_TEST_LBL  = NEW_ROOT / "test"  / "labels"

for d in [NEW_TRAIN_IMG, NEW_TRAIN_LBL,
          NEW_VALID_IMG, NEW_VALID_LBL,
          NEW_TEST_IMG,  NEW_TEST_LBL]:
    d.mkdir(parents=True, exist_ok=True)

# ── 3. GOM TẤT CẢ CẶP (ảnh, nhãn) TỪ 3 TẬP GỐC ────────────
all_pairs = []   # list of (img_path, lbl_path or None)
for img_dir, lbl_dir in SRC_SPLITS:
    for img in img_dir.iterdir():
        if img.suffix.lower() not in EXTS:
            continue
        lbl_txt = lbl_dir / f"{img.stem}.txt"
        lbl_xml = lbl_dir / f"{img.stem}.xml"
        lbl = lbl_txt if lbl_txt.exists() else (lbl_xml if lbl_xml.exists() else None)
        all_pairs.append((img, lbl))

random.seed(42)
random.shuffle(all_pairs)

total = len(all_pairs)
print(f"✅ Tổng cặp (ảnh+nhãn): {total:,}")

# ── 4. CHIA STRATIFIED THEO CÓ/KHÔNG CÓ NHÃN ───────────────
# Nếu muốn stratify theo class đầu tiên trong label, dùng labels bên dưới:
# Lấy class_id đầu tiên trong mỗi file txt làm stratify key
def get_first_class(lbl_path):
    try:
        lines = Path(lbl_path).read_text().strip().splitlines()
        return lines[0].split()[0] if lines else "0"
    except:
        return "0"

strat_labels = [get_first_class(lbl) if lbl else "no_label" for _, lbl in all_pairs]

# Split 70 / 30 trước
train_pairs, temp_pairs, train_strat, temp_strat = train_test_split(
    all_pairs, strat_labels,
    test_size=0.30, random_state=42, stratify=strat_labels
)

# Từ 30% → split 20% val và 10% test  (tức 2/3 val, 1/3 test của temp)
val_pairs, test_pairs = train_test_split(
    temp_pairs,
    test_size=1/3, random_state=42, stratify=temp_strat
)

print(f"  Train : {len(train_pairs):,}  ({len(train_pairs)/total*100:.1f}%)")
print(f"  Valid : {len(val_pairs):,}  ({len(val_pairs)/total*100:.1f}%)")
print(f"  Test  : {len(test_pairs):,}  ({len(test_pairs)/total*100:.1f}%)")

# ── 5. COPY FILE SANG THƯ MỤC MỚI ───────────────────────────
from tqdm import tqdm

def copy_split(pairs, img_dst, lbl_dst, name):
    ok_img = ok_lbl = 0
    for img_path, lbl_path in tqdm(pairs, desc=f"Copy {name}", ncols=70):
        shutil.copy2(img_path, img_dst / img_path.name)
        ok_img += 1
        if lbl_path and Path(lbl_path).exists():
            shutil.copy2(lbl_path, lbl_dst / Path(lbl_path).name)
            ok_lbl += 1
    print(f"  [{name}] ảnh={ok_img:,}  nhãn={ok_lbl:,}")

copy_split(train_pairs, NEW_TRAIN_IMG, NEW_TRAIN_LBL, "Train")
copy_split(val_pairs,   NEW_VALID_IMG, NEW_VALID_LBL, "Valid")
copy_split(test_pairs,  NEW_TEST_IMG,  NEW_TEST_LBL,  "Test")

# ── 6. VERIFY ───────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"📁 Thư mục mới: {NEW_ROOT}")
print(f"{'='*50}")
for name, idir, ldir in [
    ("Train", NEW_TRAIN_IMG, NEW_TRAIN_LBL),
    ("Valid", NEW_VALID_IMG, NEW_VALID_LBL),
    ("Test",  NEW_TEST_IMG,  NEW_TEST_LBL),
]:
    n_img = len(list(idir.glob("*")))
    n_lbl = len(list(ldir.glob("*")))
    print(f"  {name:<6}: {n_img:>6,} ảnh | {n_lbl:>6,} nhãn | {n_img/total*100:.1f}%")
print(f"{'='*50}")
print("✅ Hoàn tất! Dữ liệu gốc KHÔNG bị thay đổi.")

# ── 7. CẬP NHẬT PATH CHO CÁC CELL SAU ──────────────────────
TRAIN_IMG_DIR = NEW_TRAIN_IMG
TRAIN_LBL_DIR = NEW_TRAIN_LBL
VALID_IMG_DIR  = NEW_VALID_IMG
VALID_LBL_DIR  = NEW_VALID_LBL
TEST_IMG_DIR   = NEW_TEST_IMG
TEST_LBL_DIR   = NEW_TEST_LBL
print("✅ Paths đã cập nhật sang thư mục mới.")


# chia dữ liệu 80/10/10

In [ ]:
# ======================== CELL: CHIA LẠI DATASET CÂN BẰNG ========================
import shutil, random
from pathlib import Path
from collections import defaultdict

random.seed(42)

# ---- Đường dẫn gốc ----
DATA_ROOT = Path("D:/Code/RDD2022-1")
NEW_ROOT  = DATA_ROOT.parent / "RDD2022-balanced-10k"   # thư mục mới

SPLITS_SRC = {
    "train": (DATA_ROOT / "train/images", DATA_ROOT / "train/labels"),
    "valid": (DATA_ROOT / "valid/images", DATA_ROOT / "valid/labels"),
    "test" : (DATA_ROOT / "test/images",  DATA_ROOT / "test/labels"),
}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASS_NAMES = ["D00", "D10", "D20", "D40"]   # class id 0→D00, 1→D10, 2→D20, 3→D40

TARGET_TOTAL = 10_000
N_CLASSES    = 4
PER_CLASS    = TARGET_TOTAL // N_CLASSES      # 2500 ảnh/class

RATIO = {"train": 0.8, "valid": 0.1, "test": 0.1}

# ======================== BƯỚC 1: THU THẬP ẢNH THEO CLASS CHÍNH ========================
print("📦 Bước 1: Thu thập ảnh và phân loại theo class chính...")

class_to_imgs = defaultdict(list)   # {class_id: [(img_path, lbl_path), ...]}
multi_class   = []                   # ảnh có nhiều class — xử lý sau

for split, (img_dir, lbl_dir) in SPLITS_SRC.items():
    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in IMG_EXTS:
            continue
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue
        content = lbl_path.read_text().strip()
        if not content:
            continue   # bỏ background

        classes_in_img = set()
        for line in content.splitlines():
            parts = line.strip().split()
            if parts:
                classes_in_img.add(int(parts[0]))

        # Lấy class chính = class xuất hiện nhiều nhất trong ảnh
        class_counts = defaultdict(int)
        for line in content.splitlines():
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1
        dominant_class = max(class_counts, key=class_counts.get)
        class_to_imgs[dominant_class].append((img_path, lbl_path))

print(f"   Thống kê ảnh theo class chính:")
for cid, name in enumerate(CLASS_NAMES):
    print(f"   [{cid}] {name}: {len(class_to_imgs[cid])} ảnh")

# ======================== BƯỚC 2: SAMPLE CÂN BẰNG ========================
print(f"\n⚖️  Bước 2: Sample {PER_CLASS} ảnh/class (tổng {TARGET_TOTAL})...")

selected = []
for cid in range(N_CLASSES):
    pool = class_to_imgs[cid]
    if len(pool) < PER_CLASS:
        print(f"   ⚠️  Class {CLASS_NAMES[cid]} chỉ có {len(pool)} ảnh < {PER_CLASS} → lấy tất cả")
        sampled = pool
    else:
        sampled = random.sample(pool, PER_CLASS)
    selected.extend(sampled)

random.shuffle(selected)
print(f"   ✅ Tổng ảnh đã chọn: {len(selected)}")

# ======================== BƯỚC 3: CHIA 80/10/10 ========================
print("\n✂️  Bước 3: Chia tỉ lệ 80/10/10...")

n      = len(selected)
n_trn  = int(n * 0.8)
n_val  = int(n * 0.1)
n_test = n - n_trn - n_val

splits_new = {
    "train": selected[:n_trn],
    "valid": selected[n_trn : n_trn + n_val],
    "test" : selected[n_trn + n_val:],
}
for s, items in splits_new.items():
    print(f"   {s:6s}: {len(items)} ảnh")

# ======================== BƯỚC 4: COPY VÀO THƯ MỤC MỚI ========================
print(f"\n📂 Bước 4: Copy sang {NEW_ROOT}...")

for split, items in splits_new.items():
    img_out = NEW_ROOT / split / "images"
    lbl_out = NEW_ROOT / split / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_path, lbl_path in items:
        shutil.copy2(img_path, img_out / img_path.name)
        shutil.copy2(lbl_path, lbl_out / lbl_path.name)

    print(f"   ✅ {split}: {len(items)} ảnh + nhãn đã copy")

# ======================== BƯỚC 5: TẠO data.yaml ========================
yaml_content = f"""path: {NEW_ROOT.as_posix()}
train: train/images
val:   valid/images
test:  test/images

nc: {N_CLASSES}
names: {CLASS_NAMES}
"""
(NEW_ROOT / "data.yaml").write_text(yaml_content)
print(f"\n📄 Đã tạo: {NEW_ROOT / 'data.yaml'}")

# ======================== BƯỚC 6: KIỂM TRA KẾT QUẢ ========================
print("\n📊 KẾT QUẢ CHIA LẠI:")
print("=" * 50)
from collections import Counter

for split in ["train", "valid", "test"]:
    lbl_dir = NEW_ROOT / split / "labels"
    class_cnt = Counter()
    for lbl in lbl_dir.glob("*.txt"):
        for line in lbl.read_text().strip().splitlines():
            parts = line.strip().split()
            if parts:
                class_cnt[int(parts[0])] += 1
    total_imgs = len(list((NEW_ROOT / split / "images").iterdir()))
    print(f"\n📁 {split.upper():6s} ({total_imgs} ảnh):")
    for cid, name in enumerate(CLASS_NAMES):
        print(f"   [{cid}] {name}: {class_cnt[cid]:>5} bbox")

print("\n" + "=" * 50)
print(f"✅ Hoàn tất! Thư mục mới: {NEW_ROOT}")
print(f"   Dùng YAML: {NEW_ROOT / 'data.yaml'}")

In [ ]:
# Đếm bbox thực tế sau khi chia
from collections import Counter
from pathlib import Path

NEW_ROOT = Path("D:/Code/RDD2022-balanced-10k")
CLASS_NAMES = ["D00", "D10", "D20", "D40"]

for split in ["train", "valid", "test"]:
    cnt = Counter()
    for lbl in (NEW_ROOT / split / "labels").glob("*.txt"):
        for line in lbl.read_text().strip().splitlines():
            parts = line.strip().split()
            if parts: cnt[int(parts[0])] += 1
    total = sum(cnt.values())
    print(f"\n{split.upper()}:")
    for cid, name in enumerate(CLASS_NAMES):
        print(f"  {name}: {cnt[cid]} ({cnt[cid]/total*100:.1f}%)")